# Agente v2: corte de conocimiento, tokens y ToolMessages

La lógica está en `src/victorg_langgraph_websearchagent/agent_v2.py`.

**Conceptos antes de empezar**

- **Corte de conocimiento (julio 2025)**: el modelo (`claude-haiku-4-5`) no sabe de forma fiable lo ocurrido después. El system prompt le dice cuándo responder con conocimiento estático y cuándo buscar en la web.
- **Tokens**: los trozos de texto que cuenta (y cobra) la API. Hay de entrada (lo que se envía, incluidos los resultados de la herramienta) y de salida (lo que genera).
- **ToolMessage**: el mensaje donde LangGraph guarda lo que devuelve una herramienta. Inspeccionarlo permite comprobar si la búsqueda web estaba justificada.

## 1. Cargar el script y construir los dos agentes
El segundo es el **mismo agente sin la herramienta de búsqueda**, para comparar.

In [1]:
%load_ext autoreload
%autoreload 2

from victorg_langgraph_websearchagent.agent_v2 import (
    load_keys,
    build_search_agent,
    run_agent,
    get_answer,
    used_search,
    count_tokens,
    print_tokens,
    print_tool_messages,
    print_summary,
    compare_agents,
)

load_keys()
agent = build_search_agent()                        # con búsqueda web
agent_plain = build_search_agent(use_search=False)  # sin búsqueda web

## 2. Batería de queries
Cada query lleva el comportamiento esperado: `expected_search=False` (conocimiento estático) o `True` (necesita web).
Incluye casos límite cerca del corte (julio 2025) y una mezcla de herramientas.

In [ ]:
queries = [
    # --- Conocimiento estático / atemporal: NO debe buscar ---
    {"query": "¿Quién escribió Don Quijote de la Mancha?", "expected_search": False},
    {"query": "Explica qué es el teorema de Pitágoras", "expected_search": False},
    {"query": "¿Quién ganó el Mundial de fútbol de Qatar 2022?", "expected_search": False},
    {"query": "¿Qué es una lista por comprensión en Python?", "expected_search": False},
    # --- Herramienta multiply: NO debe buscar ---
    {"query": "Multiplica 123 por 456", "expected_search": False},
    # --- Información actual: DEBE buscar ---
    {"query": "¿Cuál es la última versión estable de LangGraph?", "expected_search": True},
    {"query": "¿Qué tiempo hace hoy en Madrid?", "expected_search": True},
    {"query": "¿Cuál es el precio actual del Bitcoin?", "expected_search": True},
    # --- Casos límite cerca del corte (julio 2025) ---
    {"query": "¿Quién ganó la Champions League 2024-2025?", "expected_search": False},
    {"query": "¿Quién ganó el Mundial de Clubes de la FIFA 2025?", "expected_search": True},
    {"query": "¿Quién ganó el Tour de Francia 2025?", "expected_search": True},
    # --- Mezcla: multiply + web ---
    {"query": "Multiplica 12 por 34 y dime cuál es la última versión estable de LangGraph", "expected_search": True},
]






## 3. Ejecutar la batería con el agente con búsqueda web

In [3]:
results = [run_agent(agent, q["query"]) for q in queries]

In [4]:
# Resumen: ¿usó la web cuando debía? tokens totales y tokens de la herramienta
print_summary(queries, results)

#   Query                                                Esperado  Usó web   Tokens   Tool  OK
1   ¿Quién escribió Don Quijote de la Mancha?            estático  no          2699      0  ✓
2   Explica qué es el teorema de Pitágoras               estático  no          2982      0  ✓
3   ¿Quién ganó el Mundial de fútbol de Qatar 2022?      estático  no          2695      0  ✓
4   ¿Qué es una lista por comprensión en Python?         estático  no          3075      0  ✓
5   Multiplica 123 por 456                               estático  no          5283     83  ✓
6   ¿Cuál es la última versión estable de LangGraph?     web       sí          8148   2664  ✓
7   ¿Qué tiempo hace hoy en Madrid?                      web       sí          8590   3190  ✓
8   ¿Cuál es el precio actual del Bitcoin?               web       sí          7810   2325  ✓
9   ¿Quién ganó la Champions League 2024-2025?           estático  sí          7932   2579  ✗
10  ¿Quién ganó el Mundial de Clubes de la FIFA 2025?    we

## 4. Inspeccionar una ejecución con búsqueda (tokens y ToolMessages)

In [5]:
i = 5  # índice de la query "última versión estable de LangGraph"
print(queries[i]["query"], "\n")
print(get_answer(results[i]))

¿Cuál es la última versión estable de LangGraph? 

Según los resultados de búsqueda, **la última versión estable de LangGraph es la 1.2.16**, lanzada el **16 de octubre de 2026**.

Algunos detalles adicionales:

- **LangGraph alcanzó v1.0** el 22 de octubre de 2025, marcando su primer lanzamiento estable importante
- La versión 1.2.7 fue lanzada el 30 de junio de 2026
- Desde entonces ha habido varias actualizaciones menores en la rama 1.2.x
- También existe una versión de TypeScript/JavaScript disponible en npm (1.4.15)

Entre las características notables añadidas en 2026 se incluyen:
- **DeltaChannel** (beta): almacena solo cambios incrementales en lugar de snapshots completos
- **Streaming API v3**: proyecciones tipadas por canal para que los consumidores se suscriban a claves de estado específicas

Puedes instalar la última versión con:
```bash
pip install langgraph==1.2.16
```


In [6]:
# Tokens totales y tokens del tool call (petición + resultado)
print_tokens(results[i])

Llamada 1 (tavily_search): input=2564 | output=84 | total=2648
Llamada 2 (respuesta final): input=5228 | output=272 | total=5500
TOTAL: input=7792 | output=356 | total=8148
TOOL CALL: petición=84 | resultado=2580 | total herramienta=2664


In [7]:
# ToolMessages en formato legible
print_tool_messages(results[i])

ToolMessage 1: tavily_search
Argumentos: {'query': 'LangGraph última versión estable 2026', 'search_depth': 'basic'}
----------------------------------------------------------------------
Query: LangGraph última versión estable 2026
Tiempo de respuesta: 0.0 s
Nº de resultados: 5

[1] LangChain vs LangGraph (2026): When to Use Which
    URL:   https://www.ayautomate.com/blog/langgraph-vs-langchain
    Score: 0.8390101
    Texto: LangGraph hit v1.0 on October 22, 2025, alongside LangChain 1.0. Current version: 1.2.7 (released June 30, 2026). Notable 2026 additions: DeltaChannel (beta), which stores only incremental state deltas rather than full snapshots on each checkpoint write, reducing storage overhead for large-state gra...

[2] langgraph · PyPI
    URL:   https://pypi.org/project/langgraph
    Score: 0.7230167
    Texto: 1.2.0a6  May 4, 2026 2 release files  Image 28  Pre-release  1.2.0a5  May 1, 2026 2 release files  Image 29  Pre-release  1.2.0a4  May 1, 2026 2 release files  Imag

In [8]:
# También como datos, por si se reutilizan después
tokens_i = count_tokens(results[i])
tokens_i["total"], tokens_i["tool_call_tokens"], tokens_i["tool_result_tokens"], tokens_i["tool_total"]

(8148, 84, 2580, 2664)

## 5. ¿Estaba justificada la búsqueda? Mismo agente con y sin herramienta
Se comparan los casos límite. Si sin web contesta bien y con seguridad, quizá la búsqueda no hacía falta.

In [9]:
borderline = [
    "¿Quién ganó la Champions League 2024-2025?",
    "¿Quién ganó el Mundial de Clubes de la FIFA 2025?",
    "¿Cuál es la última versión estable de LangGraph?",
]

comparisons = []
for question in borderline:
    comparisons.append(compare_agents(question, agent, agent_plain))
    print("\n" + "#" * 70 + "\n")

PREGUNTA: ¿Quién ganó la Champions League 2024-2025?

--- CON búsqueda web (usó web: True) ---
Según los resultados de búsqueda, **el Paris Saint-Germain (PSG)** ganó la Champions League 2024-2025.

La final se jugó el **31 de mayo de 2025** en el Allianz Arena de Múnich, Alemania, y el PSG venció al Inter de Milán con un marcador de **5-0**, estableciendo un récord de mayor goleada en una final de Champions League.

Este fue el **primer título de Champions League en la historia del PSG**, un hito histórico para el club francés.

--- SIN búsqueda web ---
No puedo confirmar con seguridad quién ganó la Champions League 2024-2025 porque esa información está fuera de mi corte de conocimiento (julio de 2025) y además es un evento que se desarrolla a lo largo de la temporada.

Aunque mi conocimiento llega hasta julio de 2025, la final de la Champions League 2024-2025 se juega típicamente en mayo o junio de 2025, por lo que podría estar cerca del límite de mi información. Sin embargo, no teng

In [10]:
# Tokens: con búsqueda vs sin búsqueda en esos mismos casos
for question, (with_s, without_s) in zip(borderline, comparisons):
    print(question)
    print("  con web:", count_tokens(with_s)["total"], "tokens | sin web:", count_tokens(without_s)["total"], "tokens")

¿Quién ganó la Champions League 2024-2025?
  con web: 7933 tokens | sin web: 1156 tokens
¿Quién ganó el Mundial de Clubes de la FIFA 2025?
  con web: 7755 tokens | sin web: 1099 tokens
¿Cuál es la última versión estable de LangGraph?
  con web: 8090 tokens | sin web: 1155 tokens
